# BDC 2026 - Stacking 5 Model DENGAN TTA (dari cache, TANPA training ulang)

Versi ini pakai cache hasil `tta_inference_5model.ipynb` (TTA di test set, dan untuk model yang
sempat dijalankan penuh, TTA juga di OOF). Sama seperti versi sebelumnya -- **tidak ada training di
notebook ini**, murni menggabungkan probabilitas yang sudah tersimpan.

Kelima cache sekarang punya format seragam (`{model}_oof_tta.npy` / `{model}_test_tta.npy` di folder
`oof_probs_tta/` dan `test_probs_tta/`), beda dari versi non-TTA sebelumnya yang formatnya beda-beda
antara 2 model pertama (object array gabungan) dan 3 model besar (file terpisah).

**5 model yang digabung:** ConvNeXtV2-Tiny, SigLIP2 ViT-B, ConvNeXtV2-Base, ConvNeXtV2-Large, SwinV2-Large
-- semua dari cache TTA.

In [1]:
import os

import joblib
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, f1_score
from sklearn.model_selection import cross_val_score

## Config

In [2]:
CONFIG = {
    "root": "BDC 2026",
    "submission_template": "BDC 2026/submission.csv",
    "submission_out": "submission_5model_stack_tta.csv",
    "solution_csv": "solution.csv",
    "meta_model_path": "meta_model_5model_tta.pkl",

    "oof_probs_dir": "oof_probs_tta",
    "test_probs_dir": "test_probs_tta",
    "model_names": [
        "convnextv2_tiny.fcmae_ft_in1k",
        "vit_base_patch16_siglip_224.v2_webli",
        "convnextv2_base.fcmae_ft_in22k_in1k",
        "convnextv2_large.fcmae_ft_in22k_in1k",
        "swinv2_large_window12to16_192to256.ms_in22k_ft_in1k",
    ],
}


## Deteksi Kelas & DataFrame

In [3]:
def get_class_order(config):
    train_dir = os.path.join(config["root"], "train")
    return sorted(os.listdir(train_dir))


def build_dataframe(train_dir, class_order):
    images, labels = [], []
    for c in class_order:
        folder = os.path.join(train_dir, c)
        for img in os.listdir(folder):
            images.append(os.path.join(folder, img))
            labels.append(c)
    df = pd.DataFrame({"image": images, "label": labels})

    label2id = {name: i for i, name in enumerate(class_order)}
    df["target"] = df["label"].map(label2id)
    return df


def build_test_dataframe(config):
    test_dir = os.path.join(config["root"], "test")
    test_images = sorted(
        os.listdir(test_dir),
        key=lambda x: int("".join(filter(str.isdigit, x)))
    )
    test_df = pd.DataFrame({"image": test_images})
    test_df["id"] = test_df["image"].apply(lambda x: int("".join(filter(str.isdigit, x))))
    return test_df

## Load Semua Cache

Sekarang formatnya seragam -- satu file `.npy` per model per jenis (OOF/test), tidak perlu lagi
gabungan 2 sumber format berbeda seperti versi non-TTA sebelumnya.

In [4]:
def load_all_caches(config):
    oof_probs_list = []
    test_probs_list = []

    for name in config["model_names"]:
        safe_name = name.replace("/", "_")
        oof = np.load(os.path.join(config["oof_probs_dir"], f"{safe_name}_oof_tta.npy"))
        test = np.load(os.path.join(config["test_probs_dir"], f"{safe_name}_test_tta.npy"))
        oof_probs_list.append(np.asarray(oof, dtype=np.float64))
        test_probs_list.append(np.asarray(test, dtype=np.float64))

    return oof_probs_list, test_probs_list, config["model_names"]


## Meta-Learner (Stacking)

In [5]:
def train_meta_learner(config, df, oof_probs_list, valid_mask_list, model_names):
    # Kalau ada model yang dilatih dengan use_kfold=False, tiap baris train cuma punya
    # OOF prediction dari model yang memang "jatah" valid split-nya. Ambil irisan baris
    # yang tervalidasi oleh SEMUA model.
    common_valid_mask = np.logical_and.reduce(valid_mask_list)
    print(f"Baris dipakai untuk melatih meta-learner: {common_valid_mask.sum()} dari {len(df)}")

    X_meta_train = np.concatenate([oof[common_valid_mask] for oof in oof_probs_list], axis=1)
    y_meta_train = df.loc[common_valid_mask, "target"].values

    meta_model = LogisticRegression(max_iter=1000)

    cv_scores = cross_val_score(meta_model, X_meta_train, y_meta_train, cv=5, scoring="f1_macro")
    print(f"Macro F1 meta-learner (5-fold CV di atas OOF probs): {cv_scores.mean():.4f} (+/- {cv_scores.std():.4f})")

    meta_model.fit(X_meta_train, y_meta_train)
    joblib.dump(meta_model, config["meta_model_path"])

    print("\nMacro F1 tiap base model secara individu (OOF):")
    for name, oof in zip(model_names, oof_probs_list):
        mask = ~np.isnan(oof).any(axis=1)
        f1 = f1_score(df.loc[mask, "target"], oof[mask].argmax(axis=1), average="macro")
        print(f"  {name}: {f1:.4f} ({mask.sum()} baris)")

    return meta_model, X_meta_train, y_meta_train

## Prediksi Akhir & Submission

In [6]:
def predict_and_submit(config, meta_model, test_df, test_probs_list):
    X_meta_test = np.concatenate(test_probs_list, axis=1)
    final_preds = meta_model.predict(X_meta_test)

    pred_map = dict(zip(test_df["id"], final_preds))

    submission = pd.read_csv(config["submission_template"])
    submission["predicted"] = submission["id"].map(pred_map)
    assert submission["predicted"].isna().sum() == 0, "Ada id yang tidak ter-mapping, cek ulang!"
    submission["predicted"] = submission["predicted"].astype(int)

    submission.to_csv(config["submission_out"], index=False)
    print(f"Submission disimpan ke: {config['submission_out']}")
    return submission

## Evaluasi (OOF & solution.csv)

In [7]:
def evaluate_oof(meta_model, X_meta_train, y_meta_train):
    oof_meta_preds = meta_model.predict(X_meta_train)
    print("=== Classification Report OOF (meta-learner, 5 model) ===")
    print(classification_report(y_meta_train, oof_meta_preds, digits=3))


def evaluate_with_solution(config, submission):
    if not os.path.exists(config["solution_csv"]):
        print(f"\n({config['solution_csv']} tidak ditemukan, skip evaluasi lokal)")
        return

    class_order = get_class_order(config)
    gt = pd.read_csv(config["solution_csv"])
    gt["predicted"] = gt["predicted"].fillna(0).astype(int)  # NaN = kelas 0 (Recyclable)
    gt = gt.rename(columns={"predicted": "true_label"})

    eval_df = submission.merge(gt, on="id", how="left")
    y_true = eval_df["true_label"]
    y_pred = eval_df["predicted"]

    print("\n=== Evaluasi vs solution.csv ===")
    print("F1 Macro:", f1_score(y_true, y_pred, average="macro"))
    print()
    print(classification_report(y_true, y_pred, target_names=class_order))
    print()
    print("Confusion matrix:")
    print(confusion_matrix(y_true, y_pred))

## Run

Load kelima cache, cek ukurannya cocok dengan data train/test saat ini (assertion, bukan asumsi diam-diam),
lalu latih meta-learner dan buat submission akhir.

In [8]:
class_order = get_class_order(CONFIG)
print("Label mapping:", {name: i for i, name in enumerate(class_order)})

train_dir = os.path.join(CONFIG["root"], "train")
df = build_dataframe(train_dir, class_order)
test_df = build_test_dataframe(CONFIG)

oof_probs_list, test_probs_list, model_names = load_all_caches(CONFIG)

# Sanity check: pastikan seluruh cache berasal dari df yang ukurannya sama persis dengan
# sekarang -- kalau tidak cocok, STOP di sini dengan pesan jelas, bukan diam-diam
# menggabungkan baris yang salah pasangan.
for name, oof in zip(model_names, oof_probs_list):
    assert oof.shape[0] == len(df), (
        f"Ukuran OOF cache '{name}' ({oof.shape[0]} baris) tidak cocok dengan df sekarang "
        f"({len(df)} baris) -- kemungkinan ada file yang ditambah/dihapus dari folder train "
        f"sejak training model ini dijalankan."
    )
for name, test_p in zip(model_names, test_probs_list):
    assert test_p.shape[0] == len(test_df), (
        f"Ukuran test cache '{name}' ({test_p.shape[0]} baris) tidak cocok dengan test_df "
        f"sekarang ({len(test_df)} baris)."
    )
print("Semua cache cocok ukurannya dengan data saat ini.\n")

valid_mask_list = [~np.isnan(oof).any(axis=1) for oof in oof_probs_list]

meta_model, X_meta_train, y_meta_train = train_meta_learner(CONFIG, df, oof_probs_list, valid_mask_list, model_names)

submission = predict_and_submit(CONFIG, meta_model, test_df, test_probs_list)

evaluate_oof(meta_model, X_meta_train, y_meta_train)
evaluate_with_solution(CONFIG, submission)

Label mapping: {'0_Recyclable': 0, '1_Electronic': 1, '2_Organic': 2}
Semua cache cocok ukurannya dengan data saat ini.

Baris dipakai untuk melatih meta-learner: 4991 dari 24954
Macro F1 meta-learner (5-fold CV di atas OOF probs): 0.9994 (+/- 0.0006)

Macro F1 tiap base model secara individu (OOF):
  convnextv2_tiny.fcmae_ft_in1k: 0.9875 (24954 baris)
  vit_base_patch16_siglip_224.v2_webli: 0.9918 (24954 baris)
  convnextv2_base.fcmae_ft_in22k_in1k: 0.9987 (4991 baris)
  convnextv2_large.fcmae_ft_in22k_in1k: 0.9989 (4991 baris)
  swinv2_large_window12to16_192to256.ms_in22k_ft_in1k: 0.9982 (4991 baris)
Submission disimpan ke: submission_5model_stack_tta.csv
=== Classification Report OOF (meta-learner, 5 model) ===
              precision    recall  f1-score   support

           0      0.998     0.999     0.999      1910
           1      1.000     1.000     1.000       790
           2      1.000     0.999     0.999      2291

    accuracy                          0.999      4991
   m